In [1]:
# NOTE: this script is designed to be run with the "arcgis_env" python environment which is simply the ArcGIS Pro python environment cloned in Anaconda.
# NOTE: if the script crashes, delete the layers that correspond to the sea level that was being processed when the crash occurred and delete everything in the temp folder. Then re-run the script.

# import packages
import numpy as np
import arcpy
import sys, datetime
from arcpy.sa import *
from arcpy.ia import *
import scipy
import scipy.special as ss
import os
import gc
import shutil

In [3]:
# Initialize ArcPy
arcpy.CheckOutExtension("Spatial") # check out Spatial Analyst
arcpy.env.overwriteOutput=True # enable overwriting existing outputs

# Set Environments
arcpy.env.workspace = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/CRC_Static_SLR/CRC_Static_SLR.gdb" # path to your geodatabase
arcpy.env.cellSize = "MINOF" # set cell size to the minimum of all rasters
arcpy.env.extent = "MINOF" # set extent to the minimum of all rasters

# directories to save files
temp_path = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/" # path to your temp folder

# Create coastline shapefile

In [3]:
## NOTE: DO NOT RUN THIS CELL IF YOU ALREADY HAVE A COASTLINE SHAPEFILE

coast_save = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/" # path to save your coastline shapefile

# ---------------------- Set Variables ---------------------- #
SLR = 0.04 # the level at which we will create the coastline. We use 0.04 m to account for displacement of MHHW from 1992 to 2005 as stated by NOAA.
name = '00ft' # name of sea level at which you will create the coastline. Your file will have this string attached in the name
islands = ['Oahu'] # name of the island you are working on. This is used to create the output file name. You can add more islands by adding them to the list, but make sure you have the corresponding DEM files in the input folder.

In [5]:
# ---------------------- Create Coastline ---------------------- 
# NOTE: this script is to be run twice for each coastline you want to create. See instructions below this cell.
now = datetime.datetime.now() # check start time
print("Ua ho'omaka i ka hola " + now.strftime('%H:%M:%S')) # print start time

# set water level and define parameters
tcari = Raster(r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/HI_TCARI_EPSG6634/Hawaii_TCARI_MHHW_EPSG6634_500m_LMSLm.tif") 
water_uncert = 0.073 # std with 1 ddof of the difference between MHHW and MSL in meters.

# ---------------------- Create Coastline ---------------------- NOTE: this script is to be run twice for each coastline you want to create. See instructions below this cell.
for island in islands: # this is set up this way in case you're creating coastlines for multiple islands. Just add more islands to the list above and make sure the directories are correct.    
    # load rasters
    
    dem = Raster(f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/HI_{island}_LMSLm_EPSG6634.tif") # path to the DEM file
    arcpy.env.cellSize = dem

    # set water level and define parameters
    waterSurface = tcari + SLR # add sea level to the TCARI surface
    dem_uncer = 0.3 # DEM uncertainty. Be careful when running multple islands because they could have different DEM uncertainties. This uncertainty is only for Matt's 2m DEM of Oahu.
    sigma_d = np.sqrt(dem_uncer**2+water_uncert**2) # cummulative uncertainty
    prob = 0.8 # probability of inundaiton (of a given area being wet)
    
    # calculate SLR inundation
    d_tilda = (waterSurface - dem) + sigma_d*np.sqrt(2)*ss.erfcinv(2*prob) # this is where the magic happens.
    # the formula above looks at every pixel and calculates the inundaiton at that pixel based on the uncertainty of the DEM and the probability of inundation.

    # identify total flood
    reclass_totalflood= Con(IsNull(d_tilda), 1, Con(d_tilda >= 0, 1)) 

    region_grp = RegionGroup(reclass_totalflood, "EIGHT", "CROSS", "", "") # select cluster of pixels with 8 neighbors to identify ocean connectivity.
    region_grp_path = f"{temp_path}{island}_{str(int(prob*100))}prob_RegGrp.tif"
    region_grp.save(region_grp_path) # save the raster

    # find region group value given to the ocean
    arcpy.management.BuildRasterAttributeTable(region_grp_path, "OVERWRITE")    # make sure the raster attribute table exists
    max_region = None
    max_count = -1
    with arcpy.da.SearchCursor(region_grp_path, ["Value", "Count"]) as cursor:
        for val, cnt in cursor:
            if cnt > max_count:
                max_count = cnt # number of pixels with a given region group value
                max_region = val # region group value

    # identify conectivity to ocean
    region_grp_path = Raster(region_grp_path)
    oceanOnlySelect = Con(region_grp_path == max_region, 1, 2) # assign value = 1 to the ocean connected region, value = 2 to all other regions. If you want to automatically delete everything that is not ocean, close parenthesis after "...max_region, 1" and write "#"

    oceanOnlySelect.save(f"{temp_path}HI_{island}_{str(int(prob*100))}prob_RegGrp_Reclass_{name}.tif") # save the raster
    oceanOnlySelect_polygName = f"{coast_save}{island}/{island}_{str(int(prob*100))}prob_{name}_coastline.shp" # name of the shapefile to save the coastline
    arcpy.RasterToPolygon_conversion(oceanOnlySelect, oceanOnlySelect_polygName,"NO_SIMPLIFY","Value", "MULTIPLE_OUTER_PART") # conversion of raster to polygon of coastline
    arcpy.Delete_management(oceanOnlySelect)
    arcpy.Delete_management(region_grp_path)
    
    del reclass_totalflood
    now = datetime.datetime.now()
    print(("pau me ka mea o " + island + " i ka hola " + now.strftime('%H:%M:%S'))) # print end time of island

now = datetime.datetime.now()
print(("pau me na mea apau i ka hola " + now.strftime('%H:%M:%S'))) # print end time of everything

Ua ho'omaka i ka hola 10:53:23
pau me ka mea o Oahu i ka hola 10:57:29
pau me na mea apau i ka hola 10:57:29


# Create Flood Maps

In [4]:
# ---------------------- Set Variables ---------------------- #
SLR = [0.04,0.3448, 0.6496, 0.9544, 1.2592, 1.564, 1.8688, 2.1736, 2.4784, 2.7832, 3.088] #  SLR increments from 0-10ft in meters + 0.04 m to account for displacement of MHHW from 1992 to 2005 as stated by NOAA
name = ['00ft','01ft','02ft','03ft','04ft','05ft','06ft','07ft','08ft','09ft','10ft'] # names for each SLR increment.
# SLR = [1.564, 1.8688, 2.1736, 2.4784, 2.7832, 3.088] #  SLR increments from 0-10ft in meters + 0.04 m to account for displacement of MHHW from 1992 to 2005 as stated by NOAA
# name = ['05ft','06ft','07ft','08ft','09ft','10ft'] # names for each SLR increment.
islands = ['Oahu']

# Constants
tcari = Raster(r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/HI_TCARI_EPSG6634/Hawaii_TCARI_MHHW_EPSG6634_500m_LMSLm.tif")
water_uncer = 0.073

prob = 0.8  # Probability of inundation (of a given area being wet)
prob_name = str(int(prob * 100)) + "prob" # name of the probability of inundation
# reclass_ranges = [[-10000, 0, 'NoData'], [0, 0.3048, 1], [0.3048, 0.6096, 2], [0.6096, 0.9144, 3], [0.9144, 1.2192, 4], [1.2192, 1.524, 5], 
#                   [1.524, 1.8288, 6], [1.8288, 2.1336, 7], [2.1336, 2.4384, 8], [2.4384, 2.7432, 9],[2.7432, 3.048, 10], [3.048, 10000, 11]]

reclass_ranges = [[-10000,0,'NoData'],[0,0.3048,1],[0.3048001,0.6096,2],[0.6096001,0.9144,3],[0.9144001,1.2192,4],[1.2192001,1.524,5], [1.524001,1.8288,6],[1.8288001,2.1336,7],[2.1336001,2.4384,8],[2.4384001,2.7432,9],[2.7432001,3.048,10],[3.048001,10000,11]]

# location of drainage shapefile
drainage_loc = f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/Oahu/Storm_Water_-_Structure_EPSG6634.shp"


In [5]:
now = datetime.datetime.now()
print(f"Ua hoomaka i ka hola {now.strftime('%H:%M:%S')}")

for island in islands:
    now = datetime.datetime.now() # check start time
    print(f"Ua ho'omaka ka modeling o {island}  i ka hola " + now.strftime('%H:%M:%S')) # print start time
    
    # Load necessary rasters and shapefiles
    dem = Raster(f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/HI_{island}_LMSLm_EPSG6634.tif")
    dem_uncer = 0.3

    arcpy.env.cellSize = dem
    arcpy.env.extent = dem
    arcpy.env.snapRaster = dem

    
    # Calculate cumulative uncertainty and set probability of inundation (0.8 = 80% prob)
    sigma_d = np.sqrt(dem_uncer**2 + water_uncer**2)

    coastline = f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/{island}_{prob_name}_MHHW_coastline.shp"
    save_path = f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Outputs/{island}/"
    # coastline_lnd = f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/{island}_{prob_name}_MHHW_coastline_lnd.shp"

    # Define paths to save files
    # nsv_path = f"C:/Users/npaoakan/Desktop/SLR_Oahu_py3_{prob_name}/TIFs/{island}/"
    # shp_path = f"C:/Users/npaoakan/Desktop/SLR_Oahu_py3_{prob_name}/SHPs/{island}/"


    for i, slr in enumerate(SLR):
        now = datetime.datetime.now()
        print(f"Ke hana nei i ka mea {name[i]} i ka hola {now.strftime('%H:%M:%S')}")

        waterSurface = tcari + slr  # Add SLR increment to level of TCARI grid

        # Calculate SLR inundation
        d_tilda = (waterSurface - dem) + sigma_d * np.sqrt(2) * ss.erfcinv(2 * prob)
        d_tilda.save(f"{temp_path}HI_{island}_{prob_name}_{name[i]}_dTilda.tif") # save d_tilda raster to temp folder to save memory
        # print('pau with calculating d_tilda and saving to temp folder')
        # create shapefile of total flood
        reclass_totalflood = Con(d_tilda >= 0, 1)
        # print('pau with reclassifying total flood on d_tilda')

        region_grp = RegionGroup(reclass_totalflood, "EIGHT", "CROSS", "", "") # select cluster of pixels with 8 neighbors to identify ocean connectivity.
        # print('pau with region regroup')
        region_grp_path = f"{temp_path}{island}_{prob_name}_{name[i]}_RegGrp.tif"
        region_grp.save(region_grp_path) # save the raster
        # print('pau with saving region regroup raster')

        oceanOnlySelectPoly_path = f"{temp_path}{island}_{prob_name}_RegGrpPoly_Reclass_{name[i]}.shp"
        region_reclass_totalflood = arcpy.RasterToPolygon_conversion(region_grp_path,
                                                                     oceanOnlySelectPoly_path,
                                                                     "NO_SIMPLIFY",
                                                                     "Value",
                                                                     "MULTIPLE_OUTER_PART")
        # print('pau with raster to polygon conversion of total flood')

        # Topographically Isolated Inundation (TII)
        arcpy.MakeFeatureLayer_management(region_reclass_totalflood,
                                          'tempName') # 'tempName' is just a placeholder for the shapefile
 
        arcpy.SelectLayerByLocation_management('tempName',
                                               "INTERSECT",
                                               coastline,
                                               None,
                                               "NEW_SELECTION", "INVERT") # select the features that are not intersecting the shoreline
        # in other words, select features that are topographically isolated
        # print('pau with selecting topographically isolated features')
        ti_inundation_shp = f"{temp_path}tiI_{island}_{prob_name}_{name[i]}.shp"
        arcpy.CopyFeatures_management('tempName',
                                      ti_inundation_shp) # save topographically isolated features to a shapefile
        # print('pau with copying topographically isolated features to shapefile')
        arcpy.DeleteFeatures_management('tempName') # delete topographically isolated features from the original shapefile

        arcpy.AddGeometryAttributes_management(ti_inundation_shp,
                                               "AREA",
                                               Area_Unit="SQUARE_METERS") # calculate area of each polygon in the shapefile
        # print('pau with adding geometry attributes to ti inundation shapefile')
        arcpy.MakeFeatureLayer_management(ti_inundation_shp,
                                          'tempName_area') # 'tempName_area' is just a placeholder for the shapefile. Needed for selecting features
        arcpy.SelectLayerByAttribute_management('tempName_area', 
                                                "NEW_SELECTION", 
                                                "POLY_AREA<=100") # select areas with less than 100 m^2. We used to use 100 pixels.
        # print('pau with selecting small polygons to delete')
        arcpy.DeleteFeatures_management('tempName_area') # delete all polygons selected polygons
        # print('pau with deleting small polygons')

        
        with arcpy.da.SearchCursor(ti_inundation_shp, "*") as cursor:
            features = [row for row in cursor] # check if there are any areas left after deleting small clusters of pixels
        # print('pau with checking if there are any topographically isolated inundation areas left')
        if not features: # if there are no TII areas then...
            print('there are no topographically isolated inundation areas')
            if name[i] == '00ft': # if name is 0p0m (which also means that there is no SCI inundation because at 0.0m of SLR the flood = the coastline) then there's no layer created
                print("a'ohe mea i loa'a mai a a'ohe flood map o ka SLR increment 0.0m")
                pass

            else: # if its any increment other than 0p0m then there is SCI inundation...
                arcpy.RepairGeometry_management(in_features=oceanOnlySelectPoly_path,
                                                delete_null="KEEP_NULL",
                                                validation_method="ESRI")
                # print('pau with repairing geometry of ocean only select polygon - no features')
                sc_inundation_shp = f"{temp_path}{island}_{prob_name}_{name[i]}_SCI.shp"
                arcpy.Erase_analysis(oceanOnlySelectPoly_path, coastline, sc_inundation_shp, '#') # erase portion of sc inundation shapefile that is oceanwards from the coastline


                # sc_inundation_shp_multi = f"{temp_path}{island}_{prob_name}_{name[i]}_SCI_multi.shp"
                # arcpy.management.MultipartToSinglepart(sc_inundation_shp, sc_inundation_shp_multi)
                # print('pau with converting sc inundation shapefile to singlepart - has features')

                
                # print('pau with erasing oceanward portions of sc inundation shapefile - no features')
                sc_inundation_raster = f"{save_path}SCI/Unclass/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      sc_inundation_raster,
                                      sc_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # clip the total flood raster to create a raster of surficially connected inundation
                # reclassify surficially connected inundation to ranges of flood depth
                # print('pau with clipping d_tilda to create sc inundation raster - no features')
                sc_inundation_raster_reclass = Reclassify(sc_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges))
                # print('pau with reclassifying sc inundation raster - no features')
                sc_inundation_raster_reclass_path = f"{temp_path}{island}_{name[i]}_SCI.tif"
                sc_inundation_raster_reclass.save(sc_inundation_raster_reclass_path) # save the reclassified raster to the temp file because it will saved with a high bit depth
                # print('pau with saving reclassified sc inundation raster to temp folder - no features')
                new_sc_inundation_raster_reclass = f"{save_path}SCI/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.CopyRaster_management(sc_inundation_raster_reclass_path, 
                                            new_sc_inundation_raster_reclass, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # change bit depth and save to the final location
                # print('pau with copying reclassified sc inundation raster to final location - no features')
                arcpy.Delete_management(sc_inundation_raster_reclass_path) # delete high bit raster to save space
                arcpy.Delete_management(sc_inundation_raster_reclass) # delete the original raster to save space
                arcpy.BuildRasterAttributeTable_management(new_sc_inundation_raster_reclass, "Overwrite")
                del sc_inundation_raster_reclass
                # print('pau with building attribute table and deleting sc inundation raster reclass - no features')

                # total inundation is only SCI inundation because there is no TII inundation
                arcpy.Copy_management(sc_inundation_shp, f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")
                # print('pau with copying sc inundation shapefile to total inundation shapefile - no features')

        else: # if there are TII areas then...
            print('there are topographically isolated inundation areas')
            if name[i] == '00ft': # if name is 0p0m (which also means that there is no SCI inundation because at 0.0m of SLR the flood = the coastline) then the TII is also the total inundation...
                temp_cl_ti_inundation_raster = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      temp_cl_ti_inundation_raster,
                                      ti_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # use the shapefile to clip the raster of topographically isolated inundation so that there are no areas with less than 100 pixels
                # print('pau with clipping d_tilda to create ti inundation raster - has features')

                # reclassify surficially connected inundation to ranges of flood depth
                ti_inundation_raster_reclass = Reclassify(temp_cl_ti_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges))
                # print('pau with reclassifying ti inundation raster - has features')

                ti_inundation_raster_reclass_path = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                ti_inundation_raster_reclass.save(ti_inundation_raster_reclass_path)
                # print('pau with saving reclassified ti inundation raster to temp folder - has features')
                # the clip tool will create a raster with a high bit depth, that is why is saved in the temp folder
                cl_ti_inundation_raster = f"{save_path}TII/{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.CopyRaster_management(ti_inundation_raster_reclass_path, 
                                            cl_ti_inundation_raster, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # copy the temporary topographically isolated inundation raster with a lower bit depth
                # print('pau with copying ti inundation raster to final location - has features')
                arcpy.Delete_management(temp_cl_ti_inundation_raster) # delete the temporary raster to save space
                arcpy.BuildRasterAttributeTable_management(cl_ti_inundation_raster, "Overwrite")
                # print('pau with building attribute table of ti inundation raster - has features')

                arcpy.Copy_management(ti_inundation_shp, f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")
                arcpy.Delete_management(ti_inundation_raster_reclass)
                del ti_inundation_raster_reclass
                # print('pau with copying ti inundation shapefile to total inundation shapefile - has features')

            else: # for all other increments (not 0p0m), there is both TII and SCI inundation so we create the total inundation shapefile with both TII and SCI
                temp_cl_ti_inundation_raster = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      temp_cl_ti_inundation_raster,
                                      ti_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # use the shapefile to clip the raster of topographically isolated inundation so that there are no areas with less than 100 pixels
                # print('pau with clipping d_tilda to create ti inundation raster - has features')

                # reclassify surficially connected inundation to ranges of flood depth
                ti_inundation_raster_reclass = Reclassify(temp_cl_ti_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges))
                # print('pau with reclassifying ti inundation raster - has features')

                ti_inundation_raster_reclass_path = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                ti_inundation_raster_reclass.save(ti_inundation_raster_reclass_path)
                # print('pau with saving reclassified ti inundation raster to temp folder - has features')

                # the clip tool will create a raster with a high bit depth, that is why is saved in the temp folder
                cl_ti_inundation_raster = f"{save_path}TII/{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.CopyRaster_management(ti_inundation_raster_reclass_path, 
                                            cl_ti_inundation_raster, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # copy the temporary topographically isolated inundation raster with a lower bit depth
                # print('pau with copying ti inundation raster to final location - has features')
                arcpy.Delete_management(temp_cl_ti_inundation_raster) # delete the temporary raster to save space
                arcpy.BuildRasterAttributeTable_management(cl_ti_inundation_raster, "Overwrite")
                # print('pau with building attribute table of ti inundation raster - has features')


                # Surficially Connected Inundation (SCI)
                arcpy.RepairGeometry_management(in_features=oceanOnlySelectPoly_path,
                                                delete_null="KEEP_NULL",
                                                validation_method="ESRI")
                
                # print('pau with repairing geometry of ocean only select polygon - has features')
                sc_inundation_shp = f"{temp_path}{island}_{prob_name}_{name[i]}_SCI.shp"
                arcpy.Erase_analysis(oceanOnlySelectPoly_path, coastline, sc_inundation_shp, '#') # erase portion of sc inundation shapefile that is oceanwards from the coastline
                print('pau with erasing oceanward portions of sc inundation shapefile - has features')



                # Surficially Connected Inundation (SCI)
                arcpy.RepairGeometry_management(in_features=sc_inundation_shp,
                                                delete_null="KEEP_NULL",
                                                validation_method="ESRI")
                print('pau with repairing geometry of sc inundation shapefile - has features')

                # sc_inundation_shp_multi = f"{temp_path}{island}_{prob_name}_{name[i]}_SCI_multi.shp"
                # arcpy.management.MultipartToSinglepart(sc_inundation_shp, sc_inundation_shp_multi)
                # print('pau with converting sc inundation shapefile to singlepart - has features')

                # d_tilda = Raster(f"{temp_path}HI_{island}_{prob_name}_{name[i]}_dTilda.tif")
                sc_inundation_raster = f"{save_path}SCI/Unclass/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      sc_inundation_raster,
                                      sc_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # clip the total flood raster to create a raster of surficially connected inundation
                print('pau with clipping d_tilda to create sc inundation raster - has features')
                
                # sc_inundation_raster_wcoast_tif = Raster(sc_inundation_raster_wcoast)
                
                # arcpy.Clip_management(sc_inundation_raster_wcoast_tif,
                #                       "#",
                #                       sc_inundation_raster,
                #                       coastline_lnd,
                #                       "#",
                #                       "ClippingGeometry",
                #                       "NO_MAINTAIN_EXTENT") # clip the total flood raster to create a raster of surficially connected inundation
                # print('pau with clipping d_tilda to create sc inundation raster - has features')
                
                
                
                # reclassify surficially connected inundation to ranges of flood depth
                sc_inundation_raster_reclass_path = f"{temp_path}{island}_{name[i]}_SCI.tif"
                sc_inundation_raster_reclass = Reclassify(sc_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges)) # save the reclassified raster to the temp file because it will saved with a high bit depth
                print('pau with reclassifying sc inundation raster - has features')

                
                sc_inundation_raster_reclass.save(sc_inundation_raster_reclass_path)
                print('pau with saving reclassified sc inundation raster to temp folder - has features')
                new_sc_inundation_raster_reclass = f"{save_path}SCI/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.CopyRaster_management(sc_inundation_raster_reclass_path, 
                                            new_sc_inundation_raster_reclass, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT")
                print('pau with copying reclassified sc inundation raster to final location - has features')
                # total inundation shapefile is created by merging the TII and SCI shapefiles
                arcpy.Merge_management([ti_inundation_shp, sc_inundation_shp], f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")
                print('pau with merging ti and sc inundation shapefiles to create total inundation shapefile')

                total_extent = f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp"
                arcpy.RepairGeometry_management(in_features=total_extent,
                                                delete_null="KEEP_NULL",
                                                validation_method="ESRI")
                print('pau with repairing geometry of total inundation shapefile')
                # total_extent.save(f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")
                # print('pau with saving total inundation shapefile to final location')

                arcpy.Delete_management(sc_inundation_raster_reclass_path) # delete high bit raster to save space
                arcpy.Delete_management(sc_inundation_raster_reclass) # delete the original raster to save space
                arcpy.BuildRasterAttributeTable_management(new_sc_inundation_raster_reclass, "Overwrite")
                del sc_inundation_raster_reclass
                print('pau with building attribute table and deleting sc inundation raster reclass - has features')


                arcpy.Delete_management(ti_inundation_raster_reclass)
                del ti_inundation_raster_reclass

        if island == 'Oahu':
            total_inundation_shp = f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp"
            total_inundation_shp_multi = f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI_multi.shp"
            arcpy.management.MultipartToSinglepart(total_inundation_shp, total_inundation_shp_multi)
            print('pau with converting total inundation shapefile to singlepart')
            # drainage_shp = f"{save_path}StormDrains/HI_{island}_{prob_name}_{name[i]}_stormDrain_intersect.shp"

            # select polygons of inundation that intersect with drainage shapefile
            arcpy.MakeFeatureLayer_management(total_inundation_shp_multi, 
                                              'tempName') # prepare the shapefile of inundation for new selection
            # select the polygons that are intersecting with the drainage shapefile
            arcpy.SelectLayerByLocation_management('tempName', 
                                                   "INTERSECT", 
                                                   drainage_loc, 
                                                   None, 
                                                   "NEW_SELECTION")
            print('pau with selecting drainage-intersecting polygons')
            with arcpy.da.SearchCursor("tempName", "*") as cursor:
                features = [row for row in cursor] # check if there are any areas left after deleting small clusters of pixels
                print(features)

            if not features:
                print(f'aohe mea no ka DBI inundation no ka {slr} SLR increment')

            else:
                drainage_flood_shp = f"{temp_path}drainage_{island}_{prob_name}_{name[i]}.shp"
                # save polygons intersecting with storm drains a new shapefile
                arcpy.CopyFeatures_management('tempName',
                                              drainage_flood_shp)
                print('pau with saving drainage-intersecting polygons to new shapefile')
        
                # select storm drains that intersect with the drainage flood shapefile
                storm_drain_instersect = f"{save_path}StormDrains/HI_{island}_{prob_name}_{name[i]}_stormDrain_intersect.shp"
                arcpy.MakeFeatureLayer_management(drainage_loc,
                                                  'tempName2') # prepare storm drain shapefile for selection of intersect with the dbi inundation shapefile
                # select storm drains that intersect with the dbi inundation shapefile
                arcpy.SelectLayerByLocation_management('tempName2', 
                                                       "INTERSECT",
                                                       drainage_flood_shp, 
                                                       None,
                                                       "NEW_SELECTION")
                print('pau with selecting storm drains that intersect with drainage-intersecting polygons')
                
                arcpy.CopyFeatures_management('tempName2',
                                              storm_drain_instersect)
                print('pau with saving storm drains that intersect with drainage-intersecting polygons to new shapefile')
                
                drainage_flood_ras_path = f"{temp_path}HI_{island}_{prob_name}_{name[i]}_DBI.tif"
                # clip the raster of total flood using the drainage flood shapefile
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      drainage_flood_ras_path,
                                      drainage_flood_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT")
                print('pau with clipping total inundation raster with drainage-intersecting polygons to create drainage-based inundation raster')
                
                drainage_flood_ras = Raster(f"{temp_path}HI_{island}_{prob_name}_{name[i]}_DBI.tif")
                drainage_flood_ras_reclass = Reclassify(drainage_flood_ras, 
                                                        "VALUE", 
                                                        RemapRange(reclass_ranges))
                print('pau with reclassifying drainage-based inundation raster to ranges of flood depth')
                temp_drainage_flood_ras = f"{temp_path}HI_{island}_{prob_name}_{name[i]}_DBI_reclass.tif"
                drainage_flood_ras_reclass.save(temp_drainage_flood_ras) # temporary save because of high bit depth
                print('pau with saving reclassified drainage-based inundation raster to temp folder')
                
                final_drainage_flood_ras = f"{save_path}DBI/HI_{island}_{prob_name}_{name[i]}_DBI.tif"
                arcpy.CopyRaster_management(temp_drainage_flood_ras, 
                                            final_drainage_flood_ras, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # final save with lower bit depth
                print('pau with saving final drainage-based inundation raster with lower bit depth')
                arcpy.Delete_management(temp_drainage_flood_ras) # delete higher bit raster to save space
                arcpy.Delete_management(drainage_flood_shp)
                arcpy.Delete_management(drainage_flood_ras_reclass) # delete the reclassified raster to save space
                del drainage_flood_ras_reclass


        # use this extent shape to intersect with drainage.

        # arcpy.management.Delete(total_ti_inundation_raster) # delete the raster that contains the pixel clusters (we only want the one that doesnt have small clusters)
        # arcpy.Delete_management(region_grp_ti_inundation_shp_path)

        if name[i] != '00ft':
            del (sc_inundation_shp, sc_inundation_raster)

        # Force garbage collection and clear ArcPy cache
        arcpy.ClearWorkspaceCache_management()
        arcpy.ClearEnvironment("workspace")
        gc.collect()

        # empty temp folder. I do this because sometimes the script crashes when there are too many files in the temp folder.
        for f in os.listdir(temp_path):
            fp = os.path.join(temp_path, f)
            try:
                if os.path.isfile(fp):
                    os.remove(fp)
            except Exception as e:
                print(f"Could not delete {fp}: {e}")


        now = datetime.datetime.now()
        print(f"Pau me ka mea {name[i]} o {island} i ka hola {now.strftime('%H:%M:%S')}")
    print(f"Pau me na mea o {island}")
print('Pau me na mea apau')

Ua hoomaka i ka hola 17:49:04
Ua ho'omaka ka modeling o Oahu  i ka hola 17:49:04
Ke hana nei i ka mea 00ft i ka hola 17:49:04
there are topographically isolated inundation areas
pau with converting total inundation shapefile to singlepart
pau with selecting drainage-intersecting polygons
[(229, (599718.1308764858, 2356389.418870563), 1235, 1235, 194504.0, 156), (250, (619557.3964014098, 2354680.574661473), 1330, 1330, 21572.0, 171), (287, (621634.0, 2354456.6896551726), 1344, 1344, 496.0, 173)]
pau with saving drainage-intersecting polygons to new shapefile
pau with selecting storm drains that intersect with drainage-intersecting polygons
pau with saving storm drains that intersect with drainage-intersecting polygons to new shapefile
pau with clipping total inundation raster with drainage-intersecting polygons to create drainage-based inundation raster
pau with reclassifying drainage-based inundation raster to ranges of flood depth
pau with saving reclassified drainage-based inundation

# Identify flooded street

In [6]:
road_1ft_depth_shp = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Outputs/Oahu/Roads/1ft/" # path to save roads with 1ft flood depth
road_2ft_depth_shp = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Outputs/Oahu/Roads/2ft/" # path to save roads with 2ft flood depth
road_shp_loc = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/Oahu/tl_rd22_hi_roads_filtered_EPSG6634.shp" # path to shapefile of roads
shore_loc = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/Oahu/Oahu_80prob_MHHW_coastline.shp" # path to shapefile of coastline created above

In [9]:
# for i in range(len(name)):
for i, name_str in enumerate(name):
    now = datetime.datetime.now()
    print((now.strftime('%H:%M:%S')))

    if i==0:
        # since at 00 ft SLR the flood = the coastline, we can just use the TII raster of 00ft SLR to create the merged raster
        ti_flood_raster = f"{save_path}TII/{island}_{prob_name}_{name_str}_TII.tif"
        merge_flood_raster = f"{temp_path}{island}_{prob_name}_00ft_full.tif"
        shutil.copyfile(ti_flood_raster, merge_flood_raster)
        total_flood_map=Raster(merge_flood_raster)
    else:
        both_flood_rasters = [
            Raster(rf"{save_path}SCI/{island}_{prob_name}_{name_str}_SCI.tif"),
            Raster(rf"{save_path}TII/{island}_{prob_name}_{name_str}_TII.tif")
        ]
        merge_flood_map = f"{island}_{prob_name}_{name_str}_full.tif"
        arcpy.management.MosaicToNewRaster(both_flood_rasters, 
                                           temp_path, 
                                           merge_flood_map, 
                                           number_of_bands='1')

        total_flood_map = Raster(rf"{temp_path}{island}_{prob_name}_{name_str}_full.tif")

    def identify_flood_above(feet):
        # reclassify the total flood raster to identify flooded streets
        total_flood_map_reclass = Reclassify(total_flood_map,
                                             "Value",
                                             RemapRange([[0, feet, 'NoData'], [feet, 12, 1]]))
        
        # create shapefile of total flood
        temp_total_flood_shp_path = f"{temp_path}totFlRec_HI_{island}_{prob_name}_{name_str}SLR_{feet}ft_strt.shp"
        temp_total_flood_shp = arcpy.RasterToPolygon_conversion(total_flood_map_reclass,
                                                                temp_total_flood_shp_path,
                                                                "NO_SIMPLIFY",
                                                                "Value")
        del total_flood_map_reclass


        # temp_total_flood_shp_no_ocean = f"{temp_path}totFlRec_HI_{island}_{prob_name}_{name_str}SLR_{feet}ft_strt_clp.shp"
        # # delete the ocean portion of the total flood shapefile
        # arcpy.analysis.Erase(temp_total_flood_shp, 
        #                      shore_loc, 
        #                      temp_total_flood_shp_no_ocean)

        # find intersection of streets with flooded areas
        flooded_street_shp = (road_1ft_depth_shp if feet == 1 else road_2ft_depth_shp) + f"{island}_{prob_name}_{name_str}SLR_{feet}ft_floodedStreet.shp"
        arcpy.analysis.Intersect([temp_total_flood_shp_path, road_shp_loc], flooded_street_shp, "ALL", "", "LINE")

    # Identify streets flooded by 1ft and 2ft
    identify_flood_above(1)
    identify_flood_above(2)

    now = datetime.datetime.now()
    print(("pau me ka mea " + name_str + " i ka hola " + now.strftime('%H:%M:%S')))

    if i != 0:
        del both_flood_rasters

09:41:17
pau me ka mea 00ft i ka hola 09:42:08
09:42:08
pau me ka mea 01ft i ka hola 09:43:22
09:43:23
pau me ka mea 02ft i ka hola 09:44:41
09:44:41
pau me ka mea 03ft i ka hola 09:46:03
09:46:03
pau me ka mea 04ft i ka hola 09:47:29
09:47:29
pau me ka mea 05ft i ka hola 09:48:52
09:48:52
pau me ka mea 06ft i ka hola 09:50:19
09:50:19
pau me ka mea 07ft i ka hola 09:51:46
09:51:46
pau me ka mea 08ft i ka hola 09:53:11
09:53:11
pau me ka mea 09ft i ka hola 09:54:35
09:54:35
pau me ka mea 10ft i ka hola 09:55:59


In [ ]:
# delete all files in the temp folder
for filename in os.listdir(temp_path):
    file_path = os.path.join(temp_path, filename)
    arcpy.Delete_management(file_path)